# LeNet 5 Paper (1998)

## Objective
1. To a gain a better understanding of LeNet image classification model version 5 introdued in the paper **[Gradient Based Learning Applied to Document Recognition](http://yann.lecun.com/exdb/publis/pdf/lecun-01a.pdf)**.

**Note**:
- To look at the entire model's implementation at one place you can have a look at the model's code written here `computer-vision/src/computer_vision/classification/lenet_5.py`
- To understand image classification thoroughly look at the information provided under the LeNet 1 notebook `computer-vision/src/computer_vision/classification/lenet_1.ipynb`
- To understand pooling layers in depth you can look at [Pooling Layers Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/pooling_layers.ipynb)
- To understand convolution neural network layer in depth you can look at [Convolution Neural Network Notebook](https://github.com/Vaibhavsareen1/pytorch-fundamentals/blob/main/notebooks/convolution_layers.ipynb) (**I advised you to go through this notebook before you go ahead for better understanding of convolution layers**)

## Lenet 5 Architecture and Training Pipeline
LeNet 5 is an improvement of LeNet 1. This model is much more closer to the modern standard template of a vision model as it introduces a separate layer for **under sampling** of features.

### Dataset
The dataset comprises of $60,000$ training and $10,000$ test samples of 10 classes which are part of the **MNIST** dataset. The samples are 1x32x32 gray scaled images of handwritten digits.

### Model Architecture
The model has 6 layers (5 part of the feature extractor block and 1 in the classifier block) as compared to only 3 layers being used in the LeNet 1 model. The paper denotes convolution layers as **Cx**, sub sampling layers as **Sx** and fully connected feedforward layers as **Fx**. The non linear activation function used with this model is **Hyperbolic Tangent** function which is applied to the output of the sub sampling layer instead of the convolution layer.

### C1 Layer
This is the first convolution layer part of the feature extractor block. This layer unlike the **h1** layer seen in Lenet 1 has 1 bias parameter assigned to each convolution filter instead of 1 bias parameter for each element in each feature map. It takes in an input of shape $(N, 1, 32, 32)$ and returns a feature map of the shape of $(N, 6, 28, 28)$. 6 convolution filters have been used in this layer where the kernel size is 5, padding is 0, stride is 1 and a bias is provided to each filter.

### S2 Layer
This is the first sub sampling layer part of the feature extractor block which follows C1 layer. In modern vision models the popular pooling/sub sampling layers are **max pooling** layers and **averegae pooling** layer. LeNet 5 used a unique sub sampling layer where instead of taking the maximum or average value it sub sampled by taking the sum of all values present in the feature map of the C1 layer. The second thing that is unique is in modern vision models trainable parameters are not assigned to the output of the sub sampling or pooling layers, but incase of LeNet 5 a set of a single weight and bias was assigned to each channel. Sub sampling convolution filter has the kernel size of 2 stride of 2 and padding 0. It takes an input of shape $(N, 6, 28, 28)$ to $(N, 6, 14, 14)$ halving the feature in a single step. These features maps are then multiplied by a single weight assigned to each channel and a bias is added which is assigned to each channel.

This layer is then passed through the **Hyperbolic Tangent** non linear activation function.

### C3 Layer
This layer is similar to H2 layer of LeNet 1 where the authors want to force different combinations of 6 feature maps to learn different types of information from them. This layer comprises of **16** convolution processes which produce a single feature map. Each convolution filter takes in an input of shape $(N, C, 14, 14)$ and produces an output of shape $(N, 1, 10, 10)$ the value of **C** is different for each convolution process. Each convolution filter has kernel size of 5, stride of 1, padding of 0 and a bias associated with it. The output of the convolution processes are then concatenated together into one output layer. The combination in this case has been provided below.

| Feature Set Number -> | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 | 11 | 12 | 13 | 14 | 15 | 16 |
| :--- | :---: | ---: | :--- | :---: | ---: | :--- | :---: | ---: |:--- | :---: | ---: | ---: | ---: | ---: | ---: | ---: |
| Feature Map 1 | X | | | | X | X | X | | | X | X | X | X | | X | X |
| Feature Map 2 | X | X | | | | X | X | X | | | X | X | X | X | | X |
| Feature Map 3 | X | X | X | | | | X | X | X | | | X | | X | X | X |
| Feature Map 4 | | X | X | X | | | X | X | X | X | | | | | X | X |
| Feature Map 5 | | | X | X | X | | | X | X | X | X | | X| X | | X |
| Feature Map 6 | | | | X | X | X | | |X | X | X | X | X | X | X | X |

During the forward pass we slice the output of S2 layer based on the above table propagate them through the convolution layers. The final output will be $(N, 16, 10, 10)$

### S4 Layer
This layer is similar to S2 layer. It takes an input of shape $(N, 16, 10, 10)$ to $(N, 16, 5, 5)$

### C5 Layer
This convolution layer converts the entire feature map into a 1 x 1 feature map. The convolution filter will be of kernel size 5, stride of 1, padding of 1 and a bias. We get 120 output features.

After convolution we flatten the feature maps into a linear layers and pass them ihrough the classifer block.

### F6 Layers
This layer has two linear neural network layers which take in 120 and 84 input features and 84 and 10 output features.

The entire model's implementation is here `computer-vision/src/computer_vision/classification/lenet_5.py`.

Below is the summary of the model

In [1]:
from torchinfo import summary
from computer_vision.classification.lenet_5 import LeNet5

summary(LeNet5(), input_size=(1, 1, 32, 32), col_names=["input_size", "output_size", "kernel_size", "num_params"])

Layer (type:depth-idx)                   Input Shape               Output Shape              Kernel Shape              Param #
LeNet5                                   [1, 1, 32, 32]            [1, 10]                   --                        44
├─Conv2d: 1-1                            [1, 1, 32, 32]            [1, 6, 28, 28]            [5, 5]                    156
├─AvgPool2d: 1-2                         [1, 6, 28, 28]            [1, 6, 14, 14]            2                         --
├─Tanh: 1-3                              [1, 6, 14, 14]            [1, 6, 14, 14]            --                        --
├─Conv2d: 1-4                            [1, 3, 14, 14]            [1, 1, 10, 10]            [5, 5]                    76
├─Conv2d: 1-5                            [1, 3, 14, 14]            [1, 1, 10, 10]            [5, 5]                    76
├─Conv2d: 1-6                            [1, 3, 14, 14]            [1, 1, 10, 10]            [5, 5]                    76
├─Conv2d: 1-7     